# (D) Telescope Resource Catalog — Reproducibility

This notebook tests whether the frozen ICARE raw capture and curated external capability CSV deterministically regenerate the official Telescope Resource Catalog. It runs the real Stage 2 and Stage 3 scripts in an isolated temporary directory, never calls the ICARE API and never executes notebook C.

## Frozen inputs and protected state

The official final artifact is fingerprinted before regeneration without loading it as a Stage 3 input. All semantic raw inputs and protected project artifacts are hashed before any subprocess runs.

In [1]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import tempfile
import time

import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 120)
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "scripts/telescopes/02_flatten.py").is_file())
CAPTURE_ID = "capture_20260808_071334"
RAW_CAPTURE = ROOT / "data/raw/telescopes/icare" / CAPTURE_ID
EXTERNAL = ROOT / "data/raw/reference/telescope_external_capabilities.csv"
OFFICIAL_INTERIM = ROOT / "data/interim/telescopes" / CAPTURE_ID
OFFICIAL_FINAL = ROOT / "data/telescope_catalog/resource_catalog.parquet"
STAGE2_SCRIPT = ROOT / "scripts/telescopes/02_flatten.py"
STAGE3_SCRIPT = ROOT / "scripts/telescopes/03_build_resource_catalog.py"
EXPECTED_FINAL_SHA = "af2fc49fccb7346c77182c8a54ded954f56e4c27a65c1dfafa8582a60ebec164"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

RAW_INPUT_NAMES = ["telescopes.json", "instruments.json", "allocations.json", "observations_page_001.json", "manifest.json"]
raw_input_paths = {name: RAW_CAPTURE / name for name in RAW_INPUT_NAMES}
raw_hashes_before = {name: sha256_file(path) for name, path in raw_input_paths.items()}
external_hash_before = sha256_file(EXTERNAL)
official_final_sha_before = sha256_file(OFFICIAL_FINAL)
official_final_size_before = OFFICIAL_FINAL.stat().st_size
official_metadata_before = pq.ParquetFile(OFFICIAL_FINAL).metadata
official_schema_before = pq.read_schema(OFFICIAL_FINAL)
PROTECTED_PATHS = [
    *raw_input_paths.values(), RAW_CAPTURE / "fetch.log", EXTERNAL,
    *[OFFICIAL_INTERIM / f"{name}.parquet" for name in ["telescopes", "instruments", "allocations", "observations"]],
    OFFICIAL_FINAL, ROOT / "notebooks/telescopes/A_eda.ipynb", ROOT / "notebooks/telescopes/B_decisions.ipynb",
    ROOT / "notebooks/telescopes/C_normalisation.ipynb", STAGE2_SCRIPT, STAGE3_SCRIPT,
]
protected_hashes_before = {str(path.relative_to(ROOT)): sha256_file(path) for path in PROTECTED_PATHS}
print(f"raw semantic inputs: {len(raw_hashes_before)}")
for name, digest in raw_hashes_before.items():
    print(f"  {name:28s} {digest}")
print(f"external CSV: {external_hash_before}")
print(f"official final: sha256={official_final_sha_before}, size={official_final_size_before:,}, rows={official_metadata_before.num_rows}, columns={official_metadata_before.num_columns}")
print(f"expected official SHA: {EXPECTED_FINAL_SHA}")

raw semantic inputs: 5
  telescopes.json              cb41047244e4b73eae6739a35248ac21f74e5d63a381a87dc48cb610e0612e92
  instruments.json             252c5f1fa9269e48f2c357157ec771062172ed45ad0b501e3b9a5bd854387878
  allocations.json             58cfab2f154a624a64c88834da844ab924442fb2cdfcf2f4b59f965e2b0d1a17
  observations_page_001.json   c152178bad11853ba475249942408517d9fff46ad6d1375445664bbf8bb2d368
  manifest.json                9a43910d389d2900f1a644602b1817f9a148dd31a05c605255427a114a9e11e1
external CSV: 6064fb0b70057ccdafb8ac669f67fed59f3d23a63b6d2704a93e87a2e7f7b63c
official final: sha256=af2fc49fccb7346c77182c8a54ded954f56e4c27a65c1dfafa8582a60ebec164, size=22,979, rows=95, columns=20
expected official SHA: af2fc49fccb7346c77182c8a54ded954f56e4c27a65c1dfafa8582a60ebec164


## Stage 2 isolated regeneration

The real flattening script reads the frozen raw capture directly and writes four new Parquet files beneath a fresh `/tmp` root. Fetching is deliberately absent.

In [2]:
TEMP_ROOT = Path(tempfile.mkdtemp(prefix="maforai_telescope_repro_"))
REGEN_INTERIM = TEMP_ROOT / "interim" / CAPTURE_ID
REGEN_FINAL = TEMP_ROOT / "final" / "resource_catalog.parquet"
stage2_command = [
    sys.executable, "-B", str(STAGE2_SCRIPT),
    "--capture-dir", str(RAW_CAPTURE),
    "--output-dir", str(REGEN_INTERIM),
]
stage2_started = time.perf_counter()
stage2_run = subprocess.run(stage2_command, cwd=ROOT, capture_output=True, text=True)
stage2_seconds = time.perf_counter() - stage2_started
print("command:", " ".join(stage2_command))
print(f"return code: {stage2_run.returncode}; duration: {stage2_seconds:.3f} s")
print("stdout tail:")
print("\n".join(stage2_run.stdout.splitlines()[-16:]))
if stage2_run.stderr.strip():
    print("stderr:")
    print(stage2_run.stderr)
if stage2_run.returncode != 0:
    raise RuntimeError(f"Stage 2 failed; isolated files retained at {TEMP_ROOT}")

command: /home/meneses/project_astronomical/MAFORAI/.venv/bin/python -B /home/meneses/project_astronomical/MAFORAI/scripts/telescopes/02_flatten.py --capture-dir /home/meneses/project_astronomical/MAFORAI/data/raw/telescopes/icare/capture_20260808_071334 --output-dir /tmp/maforai_telescope_repro_o17l4vx9/interim/capture_20260808_071334
return code: 0; duration: 6.817 s
stdout tail:
- instruments.sensitivity_data.ps1::open.limiting_magnitude: varying raw scalar types retained as JSON literal strings: float, int
- allocations.instrument.sensitivity_data.ps1::open.limiting_magnitude: varying raw scalar types retained as JSON literal strings: float, int

9. REPOSITORY SAFETY
raw files modified               : 0
existing corpus files modified   : 0
final tables/docs/notebooks created (unexpected): 0 

10. STAGE STATUS
PASS

Every control passed: the interim tables are a faithful, shape-only, provenance-aware flattening of the frozen raw capture and are safe to use as input to exploratory an

## Stage 2 comparisons

Each regenerated table is compared with its official interim counterpart for schema, dtypes, columns, row order, null positions, values and SHA-256. Official interim files are comparison targets only.

In [3]:
STAGE2_TABLES = {"telescopes": (89, "id"), "instruments": (95, "id"), "allocations": (38, "id"), "observations": (93, "id")}

def mismatch_count(left, right):
    if list(left.columns) != list(right.columns) or len(left) != len(right):
        return -1
    total = 0
    for column in left.columns:
        same = (left[column].eq(right[column]) | (left[column].isna() & right[column].isna())).fillna(False)
        total += int((~same).sum())
    return total

def compare_parquet(official_path, regenerated_path, order_key):
    official = pd.read_parquet(official_path)
    regenerated = pd.read_parquet(regenerated_path)
    official_sha, regenerated_sha = sha256_file(official_path), sha256_file(regenerated_path)
    return {
        "official": official, "regenerated": regenerated,
        "rows": len(regenerated), "columns": len(regenerated.columns),
        "schema_equal": pq.read_schema(official_path).equals(pq.read_schema(regenerated_path)),
        "dtypes_equal": official.dtypes.astype(str).to_dict() == regenerated.dtypes.astype(str).to_dict(),
        "column_order_equal": list(official.columns) == list(regenerated.columns),
        "row_order_equal": official[order_key].tolist() == regenerated[order_key].tolist(),
        "null_positions_equal": official.isna().equals(regenerated.isna()),
        "values_equal": official.equals(regenerated),
        "mismatches": mismatch_count(official, regenerated),
        "official_sha": official_sha, "regenerated_sha": regenerated_sha,
        "sha_equal": official_sha == regenerated_sha,
    }

stage2_comparisons = {}
for name, (expected_rows, order_key) in STAGE2_TABLES.items():
    stage2_comparisons[name] = compare_parquet(OFFICIAL_INTERIM / f"{name}.parquet", REGEN_INTERIM / f"{name}.parquet", order_key)
stage2_report = pd.DataFrame([{
    "table": name, "expected rows": STAGE2_TABLES[name][0], "observed rows": result["rows"],
    "schema equal": result["schema_equal"], "row order equal": result["row_order_equal"],
    "nulls equal": result["null_positions_equal"], "values equal": result["values_equal"],
    "mismatches": result["mismatches"], "SHA equal": result["sha_equal"],
} for name, result in stage2_comparisons.items()])
print(stage2_report.to_string(index=False))
stage2_semantic_pass = all(result["schema_equal"] and result["dtypes_equal"] and result["column_order_equal"] and result["row_order_equal"] and result["null_positions_equal"] and result["values_equal"] and result["mismatches"] == 0 for result in stage2_comparisons.values())
if not stage2_semantic_pass:
    raise RuntimeError(f"Stage 2 semantic comparison failed; isolated files retained at {TEMP_ROOT}")

       table  expected rows  observed rows  schema equal  row order equal  nulls equal  values equal  mismatches  SHA equal
  telescopes             89             89          True             True         True          True           0       True
 instruments             95             95          True             True         True          True           0       True
 allocations             38             38          True             True         True          True           0       True
observations             93             93          True             True         True          True           0       True


## Stage 3 isolated regeneration

The real catalog producer is invoked with explicit isolated input and output paths. Its Stage 2 input is the newly regenerated directory, and its only external input is the frozen curated CSV.

In [4]:
stage3_command = [
    sys.executable, "-B", str(STAGE3_SCRIPT),
    "--input-dir", str(REGEN_INTERIM),
    "--external-capabilities", str(EXTERNAL),
    "--output-path", str(REGEN_FINAL),
]
stage3_used_regenerated_inputs = str(REGEN_INTERIM) in stage3_command and str(OFFICIAL_INTERIM) not in stage3_command
official_final_used_as_input = str(OFFICIAL_FINAL) in stage3_command
stage3_started = time.perf_counter()
stage3_run = subprocess.run(stage3_command, cwd=ROOT, capture_output=True, text=True)
stage3_seconds = time.perf_counter() - stage3_started
print("command:", " ".join(stage3_command))
print(f"return code: {stage3_run.returncode}; duration: {stage3_seconds:.3f} s")
print("stdout tail:")
print("\n".join(stage3_run.stdout.splitlines()[-28:]))
if stage3_run.stderr.strip():
    print("stderr:")
    print(stage3_run.stderr)
if stage3_run.returncode != 0:
    raise RuntimeError(f"Stage 3 failed; isolated files retained at {TEMP_ROOT}")

command: /home/meneses/project_astronomical/MAFORAI/.venv/bin/python -B /home/meneses/project_astronomical/MAFORAI/scripts/telescopes/03_build_resource_catalog.py --input-dir /tmp/maforai_telescope_repro_o17l4vx9/interim/capture_20260808_071334 --external-capabilities /home/meneses/project_astronomical/MAFORAI/data/raw/reference/telescope_external_capabilities.csv --output-path /tmp/maforai_telescope_repro_o17l4vx9/final/resource_catalog.parquet
return code: 0; duration: 4.551 s
stdout tail:
16. PASS | SEDM Mlim UNKNOWN | Palomar 1.5m/SEDM
17. PASS | GMOS Mlim UNKNOWN | Gemini North/GMOS
18. PASS | SALT Mlim UNKNOWN | SALT/SALT
19. PASS | TAROT/TRE uses native ICARE sensitivity | 18 mag, ps1::open, 30 s
20. PASS | SVOM/VT instrument-level eligibility semantics | eligible optical imager; Mlim UNKNOWN
21. PASS | Swift/UVOTXRT instrument-level eligibility semantics | eligible optical/UV-capable imager; Mlim UNKNOWN
22. PASS | VIRT eligibility is independent of temporary status | eligible;

## Final artifact comparison

Only after Stage 3 completes is the official catalog loaded as a DataFrame for comparison. The isolated catalog is checked at byte, schema and value levels, including explicit reproduction of `telescope_diameter` and `instrument_band`, followed by catalog-specific count checks.

In [5]:
official_final_read_after_regeneration = True
catalog_comparison = compare_parquet(OFFICIAL_FINAL, REGEN_FINAL, "instrument_id")
regenerated_catalog = catalog_comparison["regenerated"]
official_catalog = catalog_comparison["official"]
known_count = int(regenerated_catalog["mlim_status"].eq("KNOWN").sum())
unknown_count = int(regenerated_catalog["mlim_status"].eq("UNKNOWN").sum())
eligible_count = int(regenerated_catalog["followup_eligible"].eq(True).sum())
ineligible_count = int(regenerated_catalog["followup_eligible"].eq(False).sum())
diameter_coverage = int(regenerated_catalog["telescope_diameter"].notna().sum())
band_coverage = int(regenerated_catalog["instrument_band"].notna().sum())
diameter_reproduced = regenerated_catalog["telescope_diameter"].equals(official_catalog["telescope_diameter"])
band_reproduced = regenerated_catalog["instrument_band"].equals(official_catalog["instrument_band"])
catalog_evidence = pd.DataFrame([
    ["rows", 95, len(regenerated_catalog)],
    ["columns", 20, len(regenerated_catalog.columns)],
    ["unique instrument IDs", 95, regenerated_catalog["instrument_id"].nunique()],
    ["telescope_diameter populated", 95, diameter_coverage],
    ["instrument_band populated", 95, band_coverage],
    ["Mlim KNOWN", 77, known_count],
    ["Mlim UNKNOWN", 18, unknown_count],
    ["followup eligible", 83, eligible_count],
    ["followup ineligible", 12, ineligible_count],
], columns=["measure", "expected", "observed"])
catalog_evidence["status"] = catalog_evidence["expected"].eq(catalog_evidence["observed"]).map({True: "PASS", False: "FAIL"})
print(catalog_evidence.to_string(index=False))
print(f"schema identical: {catalog_comparison['schema_equal']}")
print(f"dtypes identical: {catalog_comparison['dtypes_equal']}")
print(f"column order identical: {catalog_comparison['column_order_equal']}")
print(f"row order identical: {catalog_comparison['row_order_equal']}")
print(f"null positions identical: {catalog_comparison['null_positions_equal']}")
print(f"values identical: {catalog_comparison['values_equal']}")
print(f"total mismatches: {catalog_comparison['mismatches']}")
print(f"official SHA: {catalog_comparison['official_sha']}")
print(f"regenerated SHA: {catalog_comparison['regenerated_sha']}")
print(f"byte identical: {catalog_comparison['sha_equal']}")
catalog_specific_pass = (
    len(regenerated_catalog) == 95 and len(regenerated_catalog.columns) == 20
    and regenerated_catalog["instrument_id"].nunique() == 95
    and diameter_coverage == 95 and band_coverage == 95
    and diameter_reproduced and band_reproduced
    and (known_count, unknown_count) == (77, 18)
    and (eligible_count, ineligible_count) == (83, 12)
)

                     measure  expected  observed status
                        rows        95        95   PASS
                     columns        20        20   PASS
       unique instrument IDs        95        95   PASS
telescope_diameter populated        95        95   PASS
   instrument_band populated        95        95   PASS
                  Mlim KNOWN        77        77   PASS
                Mlim UNKNOWN        18        18   PASS
           followup eligible        83        83   PASS
         followup ineligible        12        12   PASS
schema identical: True
dtypes identical: True
column order identical: True
row order identical: True
null positions identical: True
values identical: True
total mismatches: 0
official SHA: af2fc49fccb7346c77182c8a54ded954f56e4c27a65c1dfafa8582a60ebec164
regenerated SHA: af2fc49fccb7346c77182c8a54ded954f56e4c27a65c1dfafa8582a60ebec164
byte identical: True


## Isolation, project safety and cleanup

The recorded Stage 3 command proves which paths were supplied. Protected project hashes are recomputed after both stages, then the isolated directory is deleted on successful comparison.

In [6]:
raw_hashes_after = {name: sha256_file(path) for name, path in raw_input_paths.items()}
external_hash_after = sha256_file(EXTERNAL)
official_final_sha_after = sha256_file(OFFICIAL_FINAL)
protected_hashes_after = {str(path.relative_to(ROOT)): sha256_file(path) for path in PROTECTED_PATHS}
raw_inputs_unchanged = raw_hashes_before == raw_hashes_after
external_unchanged = external_hash_before == external_hash_after
official_final_unchanged = official_final_sha_before == official_final_sha_after
protected_unchanged = protected_hashes_before == protected_hashes_after
all_comparisons_pass = (
    stage2_semantic_pass and stage3_run.returncode == 0
    and catalog_comparison["schema_equal"] and catalog_comparison["dtypes_equal"]
    and catalog_comparison["column_order_equal"] and catalog_comparison["row_order_equal"]
    and catalog_comparison["null_positions_equal"] and catalog_comparison["values_equal"]
    and catalog_comparison["mismatches"] == 0
    and catalog_specific_pass and catalog_comparison["sha_equal"]
    and catalog_comparison["official_sha"] == EXPECTED_FINAL_SHA
)
if all_comparisons_pass:
    shutil.rmtree(TEMP_ROOT)
temporary_run_cleaned = not TEMP_ROOT.exists()
isolation_evidence = pd.DataFrame([
    ["Stage 3 used regenerated Stage-2 inputs", True, stage3_used_regenerated_inputs],
    ["official final used as Stage-3 input", False, official_final_used_as_input],
    ["raw inputs unchanged", True, raw_inputs_unchanged],
    ["external CSV unchanged", True, external_unchanged],
    ["official final unchanged", True, official_final_unchanged],
    ["protected project files unchanged", True, protected_unchanged],
    ["temporary run cleaned", True, temporary_run_cleaned],
], columns=["measure", "expected", "observed"])
isolation_evidence["status"] = isolation_evidence["expected"].eq(isolation_evidence["observed"]).map({True: "PASS", False: "FAIL"})
print(isolation_evidence.to_string(index=False))

                                measure  expected  observed status
Stage 3 used regenerated Stage-2 inputs      True      True   PASS
   official final used as Stage-3 input     False     False   PASS
                   raw inputs unchanged      True      True   PASS
                 external CSV unchanged      True      True   PASS
               official final unchanged      True      True   PASS
      protected project files unchanged      True      True   PASS
                  temporary run cleaned      True      True   PASS


## Final reproducibility summary

The matrix consolidates execution, table equality, byte identity, semantic counts, source integrity and cleanup. Every row carries explicit expected and observed evidence.

In [7]:
matrix_rows = []
def record(number, check, expected, observed, passed):
    matrix_rows.append({"number": f"{number:02d}", "check": check, "expected": str(expected), "observed": str(observed), "status": "PASS" if bool(passed) else "FAIL"})

record(1, "frozen raw inputs unchanged", "5/5 unchanged", f"{sum(raw_hashes_before[name] == raw_hashes_after[name] for name in raw_hashes_before)}/5 unchanged", raw_inputs_unchanged)
record(2, "external CSV unchanged", external_hash_before, external_hash_after, external_unchanged)
record(3, "Stage-2 execution succeeded", "return code 0", stage2_run.returncode, stage2_run.returncode == 0)
for number, name in enumerate(["telescopes", "instruments", "allocations", "observations"], start=4):
    expected_rows = STAGE2_TABLES[name][0]
    record(number, f"regenerated {name} rows", expected_rows, stage2_comparisons[name]["rows"], stage2_comparisons[name]["rows"] == expected_rows)
for number, name in enumerate(["telescopes", "instruments", "allocations", "observations"], start=8):
    result = stage2_comparisons[name]
    semantic = result["schema_equal"] and result["dtypes_equal"] and result["column_order_equal"] and result["row_order_equal"] and result["null_positions_equal"] and result["values_equal"] and result["mismatches"] == 0
    record(number, f"{name} semantic equality", "all equal; 0 mismatches", f"equal={semantic}; mismatches={result['mismatches']}", semantic)
for number, name in enumerate(["telescopes", "instruments", "allocations", "observations"], start=12):
    result = stage2_comparisons[name]
    record(number, f"{name} SHA equality", result["official_sha"], result["regenerated_sha"], result["sha_equal"])
record(16, "Stage-3 execution succeeded", "return code 0", stage3_run.returncode, stage3_run.returncode == 0)
record(17, "regenerated catalog rows", 95, len(regenerated_catalog), len(regenerated_catalog) == 95)
record(18, "regenerated catalog schema", "schema+dtypes+20 columns equal", f"schema={catalog_comparison['schema_equal']}; dtypes={catalog_comparison['dtypes_equal']}; columns={len(regenerated_catalog.columns)}", catalog_comparison["schema_equal"] and catalog_comparison["dtypes_equal"] and len(regenerated_catalog.columns) == 20)
record(19, "telescope_diameter reproduced", "95 populated; values identical", f"populated={diameter_coverage}; identical={diameter_reproduced}", diameter_coverage == 95 and diameter_reproduced)
record(20, "instrument_band reproduced", "95 populated; values identical", f"populated={band_coverage}; identical={band_reproduced}", band_coverage == 95 and band_reproduced)
record(21, "Mlim status counts", "KNOWN=77; UNKNOWN=18", f"KNOWN={known_count}; UNKNOWN={unknown_count}", (known_count, unknown_count) == (77, 18))
record(22, "follow-up eligibility counts", "eligible=83; ineligible=12", f"eligible={eligible_count}; ineligible={ineligible_count}", (eligible_count, ineligible_count) == (83, 12))
record(23, "catalog row order equality", True, catalog_comparison["row_order_equal"], catalog_comparison["row_order_equal"])
record(24, "catalog column order equality", True, catalog_comparison["column_order_equal"], catalog_comparison["column_order_equal"])
record(25, "catalog null-position equality", True, catalog_comparison["null_positions_equal"], catalog_comparison["null_positions_equal"])
record(26, "catalog full value equality", True, catalog_comparison["values_equal"], catalog_comparison["values_equal"])
record(27, "catalog total mismatches", 0, catalog_comparison["mismatches"], catalog_comparison["mismatches"] == 0)
record(28, "catalog SHA equality", EXPECTED_FINAL_SHA, catalog_comparison["regenerated_sha"], catalog_comparison["sha_equal"] and catalog_comparison["official_sha"] == EXPECTED_FINAL_SHA)
record(29, "official final artifact unchanged", official_final_sha_before, official_final_sha_after, official_final_unchanged)
record(30, "protected project files unchanged", f"{len(PROTECTED_PATHS)}/{len(PROTECTED_PATHS)}", f"{sum(protected_hashes_before[name] == protected_hashes_after[name] for name in protected_hashes_before)}/{len(PROTECTED_PATHS)}", protected_unchanged)
isolation_pass = stage3_used_regenerated_inputs and not official_final_used_as_input and official_final_read_after_regeneration
record(31, "Stage-3 input isolation", "regenerated Stage-2 input; official final comparison-only", f"regenerated={stage3_used_regenerated_inputs}; official_input={official_final_used_as_input}; read_after={official_final_read_after_regeneration}", isolation_pass)
record(32, "temporary run cleaned", True, temporary_run_cleaned, temporary_run_cleaned)
verification = pd.DataFrame(matrix_rows)[["number", "check", "expected", "observed", "status"]]
print(verification.to_string(index=False))
pass_count = int(verification["status"].eq("PASS").sum())
fail_count = int(verification["status"].eq("FAIL").sum())
print("\n" + "=" * 60)
print("TELESCOPE RESOURCE CATALOG — REPRODUCIBILITY")
print("=" * 60)
print(f"\nFrozen inputs:\nraw capture: {RAW_CAPTURE} ({len(raw_hashes_before)} semantic files)\nexternal CSV: {EXTERNAL} ({external_hash_before})")
print(f"\nOfficial final:\npath: {OFFICIAL_FINAL}\nSHA-256: {official_final_sha_before}\nrows: {official_metadata_before.num_rows}\ncolumns: {official_metadata_before.num_columns}")
print("\nStage 2 isolated regeneration:")
for name in ["telescopes", "instruments", "allocations", "observations"]:
    result = stage2_comparisons[name]
    print(f"{name}: {result['rows']} rows | semantic_equal={result['values_equal']} | sha_equal={result['sha_equal']}")
print(f"\nStage 3 isolated regeneration:\nrows: {len(regenerated_catalog)}\ncolumns: {len(regenerated_catalog.columns)}\nMlim KNOWN: {known_count}\nMlim UNKNOWN: {unknown_count}\neligible: {eligible_count}\nineligible: {ineligible_count}")
print(f"\nFinal comparison:\nschema identical: {catalog_comparison['schema_equal']}\nrow order identical: {catalog_comparison['row_order_equal']}\nnull positions identical: {catalog_comparison['null_positions_equal']}\nvalues identical: {catalog_comparison['values_equal']}\ntotal mismatches: {catalog_comparison['mismatches']}\nofficial SHA: {catalog_comparison['official_sha']}\nregenerated SHA: {catalog_comparison['regenerated_sha']}\nbyte identical: {catalog_comparison['sha_equal']}")
print(f"\nIsolation:\nStage 3 used regenerated Stage-2 inputs: {stage3_used_regenerated_inputs}\nofficial final used as input: {official_final_used_as_input}\nexpected: no\ntemporary directory removed: {temporary_run_cleaned}")
print(f"\nVerification:\n{pass_count} PASS / {fail_count} FAIL")
print(f"\nNotebook:\ncells: 15\nexecution errors: 0")
print(f"\nOVERALL:\n{'PASS' if fail_count == 0 else 'FAIL'}")
if fail_count:
    raise RuntimeError("Reproducibility verification failed:\n" + verification.loc[verification["status"].eq("FAIL")].to_string(index=False))

number                             check                                                         expected                                                         observed status
    01       frozen raw inputs unchanged                                                    5/5 unchanged                                                    5/5 unchanged   PASS
    02            external CSV unchanged 6064fb0b70057ccdafb8ac669f67fed59f3d23a63b6d2704a93e87a2e7f7b63c 6064fb0b70057ccdafb8ac669f67fed59f3d23a63b6d2704a93e87a2e7f7b63c   PASS
    03       Stage-2 execution succeeded                                                    return code 0                                                                0   PASS
    04       regenerated telescopes rows                                                               89                                                               89   PASS
    05      regenerated instruments rows                                                               95     